In [1]:
import torch
import os
import glob
from unsloth import FastLanguageModel
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 512

In [3]:
test_dataset = load_dataset("Mathoctopus/MSVAMP", 'en', split = "test")

In [4]:
df = pd.DataFrame(test_dataset)
df[:21]


,m_query,equation,response
0,julia played tag with 18 kids on monday . she ...,- number0 number1,8.0
1,julia played tag with 11 kids on monday . she ...,- number1 number0,1.0
2,julia played tag with 5 kids on monday . she p...,- number1 number0,10.0
3,julia played tag with some kids on monday . sh...,- number1 number0,2.0
4,julia played tag with 16 kids on monday . if s...,- number0 number1,4.0
5,julia played tag with 14 kids on tuesday . if ...,+ number0 number1,22.0
6,julia played tag with 17 kids on monday 15 kid...,+ + number0 number1 number2,34.0
7,julia played tag with 15 kids on monday 18 kid...,+ number0 number1,33.0
8,julia played tag with 5 kids on monday 9 kids ...,+ number0 number2,20.0
9,julia played tag with 2 kids on monday 14 kids...,+ number1 number2,30.0


In [5]:
df.to_excel("../../msvamp_en.xlsx", index=False)

In [6]:
max_seq_length = 1024 # Choose any! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.1-8B-Instruct", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.622 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [28]:
import trl
print(trl.__file__)

/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/__init__.py


In [6]:
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
    (layers): ModuleList(
      (0-2): 3 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
     

In [7]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


def generate_response(prompt, system_prompt=None):

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    # Create input IDs + attention mask
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=2048,
        temperature=0.7,
        min_p=0.1,
        use_cache=True,
    )

    # Remove the input tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    # Decode only the generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [8]:
responses = []

while True:
    prompt = input("\nYou: ")

    if prompt.lower() in ["exit", "quit"]:
        break

    response = generate_response(prompt)
    responses.append(response)

    print("\nAssistant:", response)
    


Assistant: To find out how much Janet makes at the farmers' market, we need to calculate how many eggs she has left after eating and baking. 

Janet lays 16 eggs per day. She eats 3 and bakes 4. So, she uses 3 + 4 = 7 eggs. 

To find the number of eggs left, we subtract the number of eggs used from the total number of eggs laid. 
16 - 7 = 9 eggs left.

She sells the remaining eggs at $2 each. So, she makes 9 * 2 = $18 every day at the farmers' market.


In [9]:
def create_prompt(question):

    messages = [
        {
            "role": "user",
            "content": (
                "Solve the following math problem. "
                "Show your reasoning and give the final answer clearly.\n\n"
                f"Question:\n{question}"
            )
        }
    ]

    # Use model's chat template if available
    if tokenizer.chat_template is not None:

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    else:

        prompt = (
            "Solve the following math problem. "
            "Show your reasoning and give the final answer clearly.\n\n"
            f"Question:\n{question}\n\n"
            "Answer:"
        )

    return prompt

In [11]:
question = test_dataset[0]["m_query"]

prompt = create_prompt(question)

print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math problem. Show your reasoning and give the final answer clearly.

Question:
julia played tag with 18 kids on monday . she played tag with 10 kids on tuesday . how many more kids did she play with on monday than on tuesday ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [18]:
def collate_fn(batch):

    questions = [
        item["m_query"]
        for item in batch
    ]

    answers = [
        item["response"]
        for item in batch
    ]

    prompts = [
        create_prompt(q)
        for q in questions
    ]

    return {
        "questions": questions,
        "answers": answers,
        "prompts": prompts
    }



dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [13]:
import re

def extract_numerical_answer(response):
    # Remove <think>...</think>
    response = re.sub(
        r"<think>.*?</think>",
        "",
        response,
        flags=re.DOTALL | re.IGNORECASE
    )

    # 1. Look for \boxed{...}
    matches = re.findall(
        r"\\boxed\{\s*(-?\d+(?:\.\d+)?)\s*\}",
        response
    )
    if matches:
        return float(matches[-1])

    # 2. Look for GSM8K format: #### 10
    matches = re.findall(
        r"####\s*(-?\d+(?:\.\d+)?)",
        response
    )
    if matches:
        return float(matches[-1])

    # 3. Look for "Final Answer"
    matches = re.findall(
        r"(?:Final Answer|final answer).*?"
        r"(-?\d+(?:\.\d+)?)",
        response,
        flags=re.DOTALL
    )
    if matches:
        return float(matches[-1])

    # 4. Fallback: last number in the response
    matches = re.findall(
        r"(?<![\w.])-?\d+(?:\.\d+)?(?![\w.])",
        response
    )
    if matches:
        return float(matches[-1])

    return None

In [14]:

CHECKPOINT_EVERY = 10
CHECKPOINT_DIR = "MSVAMP_en_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ============================================================
# Find latest checkpoint
# ============================================================

checkpoint_files = glob.glob(
    os.path.join(
        CHECKPOINT_DIR,
        "checkpoint_batch_*.csv"
    )
)


if checkpoint_files:

    def get_batch_number(path):
        filename = os.path.basename(path)
        match = re.search(
            r"checkpoint_batch_(\d+)\.csv",
            filename
        )
        return int(match.group(1))

    latest_checkpoint = max(
        checkpoint_files,
        key=get_batch_number
    )

    last_batch = get_batch_number(
        latest_checkpoint
    )

    print("Latest checkpoint found:")
    print(latest_checkpoint)
    print("Last completed batch:", last_batch)

else:

    latest_checkpoint = None
    last_batch = 0

    print("No checkpoint found.")
    print("Starting evaluation from the beginning.")

No checkpoint found.
Starting evaluation from the beginning.


In [15]:
if latest_checkpoint is not None:

    checkpoint_df = pd.read_csv(
        latest_checkpoint
    )

    results = checkpoint_df.to_dict(
        orient="records"
    )

    total = len(results)

    correct = sum(
        1 for result in results
        if result["correct"]
    )

    print("Checkpoint loaded.")
    print("Already processed:", total)
    print("Already correct:", correct)

    if total > 0:
        print(
            f"Current accuracy: "
            f"{correct / total * 100:.2f}%"
        )

else:

    results = []
    correct = 0
    total = 0

In [16]:
# Number of examples already processed
start_index = total

print("Starting from example:", start_index)
print("Remaining examples:", len(test_dataset) - start_index)


# Create a subset containing only unprocessed examples
remaining_dataset = test_dataset.select(
    range(start_index, len(test_dataset))
)


dataloader = DataLoader(
    remaining_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


print("Remaining batches:", len(dataloader))

Starting from example: 0
Remaining examples: 1000
Remaining batches: 250


In [19]:
for batch_idx, batch in enumerate(dataloader):

    questions = batch["questions"]
    reference_answers = batch["answers"]
    prompts = batch["prompts"]

    # --------------------------------------------------------
    # Tokenize batch
    # --------------------------------------------------------

    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt"
    )

    # Move to GPU
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # Remove input tokens
    # --------------------------------------------------------

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[:, input_length:]

    predictions = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # --------------------------------------------------------
    # Evaluate examples
    # --------------------------------------------------------

    for question, reference_answer, prediction in zip(
        questions,
        reference_answers,
        predictions
    ):

        predicted_answer = extract_numerical_answer(
            prediction
        )

        reference_answer = float(reference_answer)

        # print(f"Question: {question}")
        # print(f"Prediction Text: {prediction}, Predicted Answer: {predicted_answer}")
        # print(f"Reference Text: {reference_text}, Reference Answer : {reference_answer}")

        is_correct = (
            predicted_answer == reference_answer
        )

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "model_output": prediction,
            "reference_answer": reference_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    current_accuracy = correct / total

    # Overall batch number
    current_batch = last_batch + batch_idx + 1

    print(
        f"Batch {current_batch} | "
        f"Examples: {total}/{len(test_dataset)} | "
        f"Accuracy: {current_accuracy * 100:.2f}%"
    )

    # --------------------------------------------------------
    # Save checkpoint every 10 batches
    # --------------------------------------------------------

    if (batch_idx + 1) % CHECKPOINT_EVERY == 0:

        checkpoint_df = pd.DataFrame(results)

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"checkpoint_batch_{current_batch}.csv"
        )

        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )

        print()
        print("=" * 60)
        print("CHECKPOINT SAVED")
        print("=" * 60)
        print("File:", checkpoint_path)
        print("Processed:", total)
        print(
            f"Accuracy: "
            f"{current_accuracy * 100:.2f}%"
        )
        print("=" * 60)
        print()

Batch 1 | Examples: 4/1000 | Accuracy: 100.00%
Batch 2 | Examples: 8/1000 | Accuracy: 100.00%
Batch 3 | Examples: 12/1000 | Accuracy: 100.00%
Batch 4 | Examples: 16/1000 | Accuracy: 100.00%
Batch 5 | Examples: 20/1000 | Accuracy: 100.00%
Batch 6 | Examples: 24/1000 | Accuracy: 95.83%
Batch 7 | Examples: 28/1000 | Accuracy: 92.86%
Batch 8 | Examples: 32/1000 | Accuracy: 93.75%
Batch 9 | Examples: 36/1000 | Accuracy: 94.44%
Batch 10 | Examples: 40/1000 | Accuracy: 95.00%

CHECKPOINT SAVED
File: MSVAMP_en_checkpoints/checkpoint_batch_10.csv
Processed: 40
Accuracy: 95.00%

Batch 11 | Examples: 44/1000 | Accuracy: 95.45%
Batch 12 | Examples: 48/1000 | Accuracy: 95.83%
Batch 13 | Examples: 52/1000 | Accuracy: 94.23%
Batch 14 | Examples: 56/1000 | Accuracy: 91.07%
Batch 15 | Examples: 60/1000 | Accuracy: 90.00%
Batch 16 | Examples: 64/1000 | Accuracy: 90.62%
Batch 17 | Examples: 68/1000 | Accuracy: 91.18%
Batch 18 | Examples: 72/1000 | Accuracy: 88.89%
Batch 19 | Examples: 76/1000 | Accuracy:

In [20]:
accuracy = correct / total

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(f"Correct : {correct}")
print(f"Total   : {total}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

FINAL RESULTS
Correct : 805
Total   : 1000
Accuracy: 0.8050
Accuracy: 80.50%
